In [2]:
import yfinance as yf
import datetime as dt

date = dt.datetime(2013, 8, 28)
ticker = "^GSPC"
sp500 = yf.Ticker(ticker)

sp500_history = sp500.history(start=date - dt.timedelta(days=365), end=date + dt.timedelta(days=1))
St = sp500_history["Close"].iloc[-1]

In [3]:
# Long Position with initial value xSt = 1000000
xSt = 1000000

x = xSt / St
print(x)

611.6357732862103


In [4]:
from scipy.stats import norm
# 99 CL for Standard Normal CDF
alpha = 0.99
z = norm.ppf(1-alpha)
print(z)

-2.3263478740408408


In [7]:
import pandas as pd

prices = sp500_history["Close"]

returns = prices.pct_change()

df = pd.DataFrame({
    "Price": prices, 
    "Returns": returns
})

df.index = df.index.date

df = df.dropna()

df.head()

,Price,Returns
2012-08-29,1410.489990,0.000844
2012-08-30,1399.479980,-0.007806
2012-08-31,1406.579956,0.005073
2012-09-04,1404.939941,-0.001166
2012-09-05,1403.439941,-0.001068


In [18]:
df

,Price,Returns,EWMA_Var
2012-08-29,1410.489990,0.000844,7.129256e-07
2012-08-30,1399.479980,-0.007806,4.325986e-06
2012-08-31,1406.579956,0.005073,5.610726e-06
2012-09-04,1404.939941,-0.001166,5.355650e-06
2012-09-05,1403.439941,-0.001068,5.102705e-06
...,...,...,...
2013-08-22,1656.959961,0.008619,3.987724e-05
2013-08-23,1663.500000,0.003947,3.841934e-05
2013-08-26,1656.780029,-0.004040,3.709331e-05
2013-08-27,1630.479980,-0.015874,4.998711e-05


In [20]:
4.744086 * 10 ** (-5)


4.7440860000000006e-05

In [23]:
import numpy as np

In [28]:
# Method 1: Calculate the EWMA of returns by hand

lambda_ = 0.94

ewma_var = df.iloc[0, 1] ** 2

emwa_var_list = [ewma_var]

for i in range(1, df.shape[0]):
    ewma_var = lambda_ * ewma_var + (1 - lambda_) * df["Returns"].iloc[i] ** 2
    emwa_var_list.append(ewma_var)

df["EWMA_Var"] = emwa_var_list

print("EWMA Variance:", ewma_var)

ewma_vol_daily = np.sqrt(ewma_var)
print("EWMA_vol_daily:", ewma_vol_daily)

ewma_vol_annual = ewma_vol_daily * np.sqrt(252)
print("EWMA Vol Annual:", ewma_vol_annual)

EWMA Variance: 4.7440859486309475e-05
EWMA_vol_daily: 0.006887732535915537
EWMA Vol Annual: 0.10933936432296462


In [34]:
df

,Price,Returns,EWMA_Var
2012-08-29,1410.489990,0.000844,7.129256e-07
2012-08-30,1399.479980,-0.007806,4.325986e-06
2012-08-31,1406.579956,0.005073,5.610726e-06
2012-09-04,1404.939941,-0.001166,5.355650e-06
2012-09-05,1403.439941,-0.001068,5.102705e-06
...,...,...,...
2013-08-22,1656.959961,0.008619,3.987724e-05
2013-08-23,1663.500000,0.003947,3.841934e-05
2013-08-26,1656.780029,-0.004040,3.709331e-05
2013-08-27,1630.479980,-0.015874,4.998711e-05


In [ ]:
# Method 2: Use pandas ewm function to calculate the EWMA of returns

date_ewma_var = df["Returns"].ewm(alpha = 1-lambda_, adjust=False).var().iloc[-1]
date_ewma_var

4.770163388550673e-05

# Parametric VaR

**VaR scenario in log return terms**: at 99 percent CL, use 0.01 quantile of $r_{t, t+\tau}$ $z_{1-\alpha}\sigma_t\sqrt{\tau}$, the $1-\alpha$ quantile of $r_{t, t+\tau}$

In [36]:
z * ewma_vol_daily

-0.01602326194188904

A bit over -1.5%.

**Change in risk factor in VaR scenario**: convert log return into arithmetic return to compute PnL. The $1-\alpha$ quantile of $S_{t+\tau} - S_t$ is: 

\begin{equation*}
S_te^{z_{1-\alpha}\sigma_t\sqrt{\tau}} - S_t = S_t(e^{z_{1-\alpha}\sigma_t\sqrt{\tau}}-1)

\end{equation*}

0.01-quantile of 1 day change

In [37]:
St * np.exp(z * ewma_vol_daily) - St

-25.98862441544088

**VaR scenario in PnL terms**: $1-\alpha$ quantile change in pos value: $xS_t(e^{z_{1-\alpha}\sigma_t\sqrt{\tau}}-1)$

In [41]:
x * (St * np.exp(z * ewma_vol_daily) - St)

-15895.57239098307

**VaR** at confidence level $\alpha$ is PnL quantile as a pos number:

\begin{equation*}
VaR_{t}(\alpha, \tau) = -xS_t(e^{z_{1-\alpha}\sigma_t\sqrt{\tau}}-1) = xS_t(1-e^{z_{1-\alpha}\sigma_t\sqrt{\tau}})
\end{equation*}

At 99% CL the daily VAR is $15896.

In [42]:
- x * (St * np.exp(z * ewma_vol_daily) - St)

15895.57239098307

In [43]:
- x * (St * np.exp(np.sqrt(5)* z * ewma_vol_daily) - St)

35194.838223347666

# Monte Carlo